# 02. clean

## 0. setup

In [3]:
import re

import gc
from pathlib import Path

import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

data_interim.mkdir(parents=True, exist_ok=True)
data_proc.mkdir(parents=True, exist_ok=True)

## 1. config

In [4]:
# cartel
in_ref = data_raw / 'cartel' / 'manual' / 'cartel_manual_v7.xlsx'
eea_membership_ref = 'infr_start'     # 'infr_start' | 'decision' : ref year for bloc membership

#  patents
in_pat = data_interim / 'pat_data.parquet'
in_ipc_isic = data_raw / 'external' / 'ipc4_to_isic_rev4_3.txt'

# outputs
out_treated   = data_proc / 'cartel_treated.parquet'    # infringement x nace3 x ctry
out_patpanel  = data_interim / 'pat_panel.parquet'      # ctry x isic3 x year
out_coarse    = data_interim / 'nace_coarse_review.xlsx'
out_infr      = data_proc / 'clean_infringements.xlsx'

print(f'cartel : {in_ref.name} -> {out_treated.name}')
print(f'patents: {in_pat.name} + {in_ipc_isic.name} -> {out_patpanel.name}')

cartel : cartel_manual_v7.xlsx -> cartel_treated.parquet
patents: pat_data.parquet + ipc4_to_isic_rev4_3.txt -> pat_panel.parquet


## 2. country scope

In [5]:
# bloc membership for cartel scope expansion
ref_ctry = pd.read_excel(in_ref, sheet_name='ref_ctry_timeline')
universe = set(ref_ctry['ctry_iso2'])

def members(bloc, year):
    '''ISO2s that are members of `bloc` ('EEA'|'EU') in `year`.'''
    if pd.isna(year):
        year = 3000  # ref year unknown -> include everyone ever a member
    entry = f'{bloc.lower()}_entry_year'
    exit_ = f'{bloc.lower()}_exit_year'
    m = ref_ctry[(ref_ctry[entry] <= year) &
                 (ref_ctry[exit_].isna() | (year < ref_ctry[exit_]))]
    return set(m['ctry_iso2'])

# patent country universe
europe_iso2 = {
    # EU27
    'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR','DE','GR','HU','IE',
    'IT','LV','LT','LU','MT','NL','PL','PT','RO','SK','SI','ES','SE',
    'IS','LI','NO','CH',                    # EFTA / EEA
    'GB',                                   # UK
    'AD','MC','SM','VA',                    # microstates
    'AL','BA','MK','ME','RS','XK',          # Western Balkans
    'BY','MD','UA','RU',                    # Eastern Europe
    'TR','GE','AM','AZ',                    # transcontinental / borderline
}

print(f'bloc universe: {len(universe)} | patent (europe): {len(europe_iso2)} | timeline rows {ref_ctry.shape[0]}')

bloc universe: 31 | patent (europe): 50 | timeline rows 31


## 3. cartel

### 3.1 filter infringements

In [6]:
cartels   = pd.read_excel(in_ref, sheet_name='cartels')
decisions = pd.read_excel(in_ref, sheet_name='decisions')
print('cartels', cartels.shape, '| decisions', decisions.shape)

# keep only usable, non-excluded infringements
infr = cartels[cartels['exclude_reason'].isna()].copy()
for col in ['decision_year', 'start_year', 'end_year']:
    infr[col] = pd.to_numeric(infr[col], errors='coerce').astype('Int64')

print(f'kept after exclude_reason: {len(infr)}')
print('decision-year span :', int(infr['decision_year'].min()), '-', int(infr['decision_year'].max()))
print('formation-year span:', int(infr['start_year'].min()),   '-', int(infr['start_year'].max()))
print('breakup-year span  :', int(infr['end_year'].min()),     '-', int(infr['end_year'].max()))

cartels (215, 23) | decisions (199, 10)
kept after exclude_reason: 164
decision-year span : 1982 - 2025
formation-year span: 1969 - 2018
breakup-year span  : 1976 - 2022


### 3.2 case-level fine
Amendments / re-adoptions were entered as additive deltas, so the per-case **sum** is the final fine (`min_count=1` keeps a genuine no-fine case as NaN, not 0).

In [7]:
decisions['fine_eur'] = pd.to_numeric(decisions['fine_eur'], errors='coerce')

fine_by_case = (decisions
                .groupby('case_id')['fine_eur']
                .sum(min_count=1)
                .rename('fine_eur')
                .reset_index())

legal_by_case = (decisions.sort_values(['case_id', 'decision_seq'])
                 .groupby('case_id', as_index=False)
                 .first()[['case_id', 'legal_basis']])

has_amend = decisions.loc[
    decisions['decision_type'].isin(['amendment', 're-adoption']), 'case_id'].unique()

fine_info = fine_by_case.merge(legal_by_case, on='case_id', how='left')
fine_info['has_amendment'] = fine_info['case_id'].isin(has_amend).astype(int)

infr = infr.merge(fine_info, on='case_id', how='left')

### 3.3 parse NACE and explode to (infringement × NACE4)

In [8]:
def split_nace(cell):
    if not isinstance(cell, str):
        return []
    out = []
    for tok in cell.split(','):
        digits = re.sub(r'[^0-9]', '', tok)
        if len(digits) >= 2:
            out.append(digits)
    return out

infr['nace_list'] = infr['nace_code'].apply(split_nace)

nace_long = (infr.explode('nace_list')
             .rename(columns={'nace_list': 'nace'})
             .dropna(subset=['nace']))
nace_long['nace_level'] = nace_long['nace'].str.len()

# analysis grain is 3-digit (ISIC3 = NACE group). Anything >=3 digits resolves;
# 2-digit codes are too coarse for the grain and go to review.
nace_long['ind'] = np.where(nace_long['nace_level'] >= 3,
                            nace_long['nace'].str[:3], pd.NA)

# k = number of DISTINCT 3-digit industries a cartel touches (denominator for the 1/k split)
k_ind = (nace_long[nace_long['ind'].notna()]
         .groupby('infringement_id')['ind'].nunique()
         .rename('k_ind'))
nace_long = nace_long.merge(k_ind, on='infringement_id', how='left')

coarse   = nace_long[nace_long['nace_level'] < 3].copy()   # 2-digit only -> review
ind_long = (nace_long[nace_long['ind'].notna()]
            .drop_duplicates(['infringement_id', 'ind'])   # 2451 & 2452 -> one 245 row
            .copy())

print('clean 3-digit rows:', len(ind_long),
      '| distinct ISIC3:', ind_long['ind'].nunique(),
      '| distinct treated infringements:', ind_long['infringement_id'].nunique())
print('coarse (2-digit) rows routed to review:', len(coarse)) 

clean 3-digit rows: 166 | distinct ISIC3: 59 | distinct treated infringements: 146
coarse (2-digit) rows routed to review: 18


### 3.4 expand geographic scope to (infringement × country)

In [9]:
def ref_year(row):
    return row['start_year'] if (eea_membership_ref == 'infr_start'
                                 and pd.notna(row['start_year'])) else row['decision_year']

def scope_for(row):
    '''Return (set_of_iso2, scope_source) for one infringement.'''
    raw = row['ctry_iso2']
    yr = ref_year(row)
    if not isinstance(raw, str) or not raw.strip():
        return set(), 'missing'                 # no explicit scope -> contributes no cells
    toks = [t.strip().upper() for t in raw.split(',') if t.strip()]
    out, src = set(), 'explicit'
    for t in toks:
        if t in ('EEA', 'EU'):
            out |= members(t, yr); src = 'bloc_expanded'
        elif t in universe:
            out.add(t)
        # else: non-panel token (JP/CH/KR/...) -> ignored
    if not out:
        return set(), 'non_panel_only'
    return out, src

scope = infr.apply(scope_for, axis=1)
infr['ctry_set']     = scope.apply(lambda x: sorted(x[0]))
infr['scope_source'] = scope.apply(lambda x: x[1])

ctry_long = (infr[['infringement_id', 'ctry_set', 'scope_source']]
             .explode('ctry_set')
             .rename(columns={'ctry_set': 'ctry_iso2'})
             .dropna(subset=['ctry_iso2']))

print(infr['scope_source'].value_counts(dropna=False).to_string())
print('\n(infringement x country) rows:', len(ctry_long))

scope_source
missing           96
explicit          58
bloc_expanded      9
non_panel_only     1

(infringement x country) rows: 459


### 3.5 build atomic treated table: (infringement × NACE4 × country)
One row per treated triple. Candidate cohorts (decision/start/end) carried side by side; **no first-treat collapse here** — that is a modelling choice made at the analysis grain in `03_panel`.

In [10]:
keep_cols = ['infringement_id', 'case_id', 'cartel_name', 'case_structure',
             'segment_seq', 'decision_year', 'start_year', 'end_year',
             'fine_eur', 'has_amendment', 'k_ind']
treated = (ind_long[keep_cols + ['ind']]
           .merge(ctry_long, on='infringement_id', how='inner')
           .drop_duplicates(['infringement_id', 'ind', 'ctry_iso2']))

treated = treated.rename(columns={'decision_year': 'cohort_decision',
                                  'start_year':    'cohort_start',
                                  'end_year':      'cohort_end'})
# alias case_id as the event id used by the repeat-treatment guard in 03_panel:
# multiple infringement_ids sharing a case_id = ONE enforcement event, not several
treated['enforcement_event_id'] = treated['case_id']

print('treated rows (infr x isic3 x ctry):', len(treated))
print('distinct treated ISIC3:', treated['ind'].nunique())
print('distinct (isic3 x ctry) cells:', treated[['ind', 'ctry_iso2']].drop_duplicates().shape[0])

treated rows (infr x isic3 x ctry): 443
distinct treated ISIC3: 34
distinct (isic3 x ctry) cells: 276


### 3.6 diagnostics

In [14]:
print('=== scope sources ===')
for pol in ['explicit', 'bloc_expanded', 'inferred_eea_missing', 'missing', 'non_panel_only']:
    n = (infr['scope_source'] == pol).sum()
    if n: print(f'  {pol:<22}: {n} infringements')

print('\n=== fine coverage ===')
print(f'  infringements with a fine : {infr['fine_eur'].notna().sum()} / {len(infr)}')
print(f'  with amendment/re-adoption: {int(infr['has_amendment'].fillna(0).sum())}')

print('\n=== treated grain ===')
print('  distinct treated NACE3      :', treated['ind'].nunique())
print('  distinct (nace3 x ctry)     :', treated[['ind','ctry_iso2']].drop_duplicates().shape[0])
print('  multi-NACE cartels (k>1)    :', (treated.drop_duplicates('infringement_id')['k_ind'] > 1).sum())

=== scope sources ===
  explicit              : 58 infringements
  bloc_expanded         : 9 infringements
  missing               : 96 infringements
  non_panel_only        : 1 infringements

=== fine coverage ===
  infringements with a fine : 164 / 164
  with amendment/re-adoption: 18

=== treated grain ===
  distinct treated NACE3      : 34
  distinct (nace3 x ctry)     : 276
  multi-NACE cartels (k>1)    : 13


### 3.7 save

In [15]:
treated.to_parquet(out_treated, index=False)

infr_out = infr.drop(columns=['nace_list', 'ctry_set'])
infr_out.to_excel(out_infr, index=False)
coarse.to_excel(out_coarse, index=False)

print(f'saved: {out_treated.name} ({len(treated):,} rows), {out_infr.name}, {out_coarse.name}')

saved: cartel_treated.parquet (443 rows), clean_infringements.xlsx, nace_coarse_review.xlsx


## 4. patents

### 4.1 concordance to industry key (ISIC rev4, 3-digit)

In [16]:
conc = pd.read_csv(in_ipc_isic, dtype=str)
conc.columns = ['ipc_sub', 'ind', 'w']            # ipc4, isic_rev4_3, probability_weight
conc['w'] = conc['w'].astype(float)

wsum = conc.groupby('ipc_sub')['w'].transform('sum')
conc['w'] = conc['w'] / wsum
print(f'concordance rows: {len(conc):,} | IPC subclasses: {conc['ipc_sub'].nunique():,} '
      f'| industries: {conc['ind'].nunique():,}')

concordance rows: 2,932 | IPC subclasses: 636 | industries: 212


### 4.2 patents to IPC subclass + fractional weight

In [17]:
pat = pd.read_parquet(in_pat)
pat = pat.dropna(subset=['appln_id', 'app_year', 'ipc', 'ctry_code', 'app_share'])
pat['app_year']  = pat['app_year'].astype('int32')
pat['ipc_sub']   = pat['ipc'].astype(str).str[:4].str.strip()
pat['ctry_iso']  = pat['ctry_code'].astype(str).str.upper().str.strip()
pat['app_share'] = pd.to_numeric(pat['app_share'], errors='coerce')

# each (patent x applicant) spreads equally across its distinct IPC subclasses
pat = pat.drop_duplicates(['appln_id', 'applt_id', 'ipc_sub'])
n_sub = pat.groupby(['appln_id', 'applt_id'])['ipc_sub'].transform('count')
pat['frac_weight'] = (pat['app_share'] / n_sub).astype('float64')

print(f'patent rows: {len(pat):,} | applications: {pat['appln_id'].nunique():,}')

patent rows: 13,762,762 | applications: 6,880,084


### 4.3 assign to industry

In [18]:
matched = pat['ipc_sub'].isin(set(conc['ipc_sub']))
print(f'IPC-subclass rows matched to concordance: {matched.mean():.1%} '
      f'({(~matched).sum():,} unmatched rows dropped)')

# collapse to (ctry, year, ipc_sub) then expand to industry via concordance weights
ipc_cell = (pat.groupby(['ctry_iso', 'app_year', 'ipc_sub'], observed=True)['frac_weight']
              .sum().reset_index())
ipc_cell = ipc_cell.merge(conc, on='ipc_sub', how='inner')
ipc_cell['iw'] = (ipc_cell['frac_weight'] * ipc_cell['w']).astype('float32')

panel = (ipc_cell.groupby(['ctry_iso', 'ind', 'app_year'], observed=True)['iw']
         .sum().reset_index()
         .rename(columns={'iw': 'pat_frac', 'app_year': 'year'}))
del ipc_cell

# distinct counts via modal (top-weight) industry per IPC subclass
top_ind = (conc.sort_values('w').drop_duplicates('ipc_sub', keep='last')
           .set_index('ipc_sub')['ind'])
pat['ind_top'] = pat['ipc_sub'].map(top_ind)
counts = (pat.dropna(subset=['ind_top'])
          .groupby(['ctry_iso', 'ind_top', 'app_year'], observed=True)
          .agg(n_pat_appln=('appln_id', 'nunique'),
               n_applt    =('applt_id', 'nunique'))
          .reset_index().rename(columns={'ind_top': 'ind', 'app_year': 'year'}))

panel = panel.merge(counts, on=['ctry_iso', 'ind', 'year'], how='left')
panel[['n_pat_appln', 'n_applt']] = panel[['n_pat_appln', 'n_applt']].fillna(0).astype(int)

IPC-subclass rows matched to concordance: 98.7% (181,627 unmatched rows dropped)


### 4.4 restrict to country scope

In [19]:
n0 = len(panel)
panel = panel[panel['ctry_iso'].isin(europe_iso2)].copy()
print(f'country filter to EU/EEA universe: {n0:,} -> {len(panel):,} cells')
print(f'panel: {len(panel):,} cell-years | {panel['ind'].nunique()} industries | '
      f'{panel['ctry_iso'].nunique()} countries | {panel['year'].min()}-{panel['year'].max()}')

country filter to EU/EEA universe: 387,373 -> 200,191 cells
panel: 200,191 cell-years | 212 industries | 49 countries | 1978-2024


### 4.5 save

In [20]:
panel.to_parquet(out_patpanel, index=False)
print(f'saved: {out_patpanel.name} ({len(panel):,} rows)')

del pat, panel; gc.collect()

saved: pat_panel.parquet (200,191 rows)


10549